# ALTRIX ML Model Training
## XGBoost + Isolation Forest with SMOTE Balancing

This notebook trains the insider threat detection models using CERT r4.2 dataset.

In [12]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, RepeatedStratifiedKFold, cross_validate
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score, average_precision_score
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import pickle
import os

print("Libraries imported successfully")

Libraries imported successfully


## 1. Load CERT r4.2 Data

In [17]:
# Load datasets
data_path = '../data/cert_r4.2/'

try:
    logon_df = pd.read_csv(data_path + 'logon.csv')
    device_df = pd.read_csv(data_path + 'device.csv')
    email_df = pd.read_csv(data_path + 'email.csv')
    file_df = pd.read_csv(data_path + 'file.csv')
    http_df = pd.read_csv(data_path + 'http.csv')
    insiders_df = pd.read_csv(data_path + 'insiders.csv')

    # Build malicious user labels from insiders.csv (source of truth)
    MALICIOUS_USERS = sorted(insiders_df['user'].dropna().astype(str).unique().tolist())
    INSIDER_SCENARIO_BY_USER = dict(zip(insiders_df['user'], insiders_df['scenario']))

    print("Data loaded successfully:")
    print(f"  Logon events: {len(logon_df)}")
    print(f"  Device events: {len(device_df)}")
    print(f"  Email events: {len(email_df)}")
    print(f"  File events: {len(file_df)}")
    print(f"  HTTP events: {len(http_df)}")
    print(f"  Insider labels: {len(MALICIOUS_USERS)} users")
    print(f"  Insider scenarios: {sorted(insiders_df['scenario'].dropna().unique().tolist())}")
except Exception as e:
    print(f"ERROR: Could not load CERT data. Please ensure data is in {data_path}")
    print(f"Error: {e}")

Data loaded successfully:
  Logon events: 223089
  Device events: 10008
  Email events: 356080
  File events: 483061
  HTTP events: 1858792
  Insider labels: 10 users
  Insider scenarios: ['Data Exfiltration']


## 2. Feature Engineering

In [18]:
def extract_features(user_id):
    """Extract behavioral features for a user"""
    features = {}
    
    # Logon features
    user_logons = logon_df[logon_df['user'] == user_id]
    features['logon_count'] = len(user_logons)
    features['failed_logon_count'] = len(user_logons[user_logons['activity'] == 'Logon']) if 'activity' in user_logons.columns else 0
    features['after_hours_logon'] = len(user_logons) * 0.2  # Approximate
    
    # Device (USB) features
    user_devices = device_df[device_df['user'] == user_id]
    features['usb_connect_count'] = len(user_devices[user_devices['activity'] == 'Connect']) if 'activity' in user_devices.columns else len(user_devices) * 0.5
    features['usb_disconnect_count'] = len(user_devices[user_devices['activity'] == 'Disconnect']) if 'activity' in user_devices.columns else len(user_devices) * 0.5
    
    # Email features
    user_emails = email_df[email_df['user'] == user_id]
    features['email_sent_count'] = len(user_emails)
    if 'attachments' in user_emails.columns:
        attachments_numeric = pd.to_numeric(user_emails['attachments'], errors='coerce').fillna(0)
        features['email_with_attachment'] = int((attachments_numeric > 0).sum())
    else:
        features['email_with_attachment'] = int(len(user_emails) * 0.3)
    features['external_email_count'] = len(user_emails) * 0.4  # Approximate
    
    # File features
    user_files = file_df[file_df['user'] == user_id]
    features['file_access_count'] = len(user_files)
    features['file_copy_count'] = len(user_files[user_files['activity'] == 'File Copy']) if 'activity' in user_files.columns else 0
    
    # HTTP features
    user_http = http_df[http_df['user'] == user_id]
    features['http_request_count'] = len(user_http)
    features['job_search_visits'] = 0  # Would need URL analysis
    
    # Label
    features['is_malicious'] = 1 if user_id in MALICIOUS_USERS else 0
    
    return features

# Get unique users
all_users = set()
if 'user' in logon_df.columns:
    all_users.update(logon_df['user'].unique())
if 'user' in device_df.columns:
    all_users.update(device_df['user'].unique())
if 'user' in email_df.columns:
    all_users.update(email_df['user'].unique())
if 'user' in file_df.columns:
    all_users.update(file_df['user'].unique())
if 'user' in http_df.columns:
    all_users.update(http_df['user'].unique())

print(f"Total unique users: {len(all_users)}")

# Extract features for all users
user_features = []
for user in all_users:
    user_features.append(extract_features(user))

# Create dataframe
features_df = pd.DataFrame(user_features)
print(f"\nFeatures extracted for {len(features_df)} users")
print(f"Malicious users: {features_df['is_malicious'].sum()}")
print(f"Normal users: {len(features_df) - features_df['is_malicious'].sum()}")
print(f"\nFeature columns: {list(features_df.columns)}")

Total unique users: 200

Features extracted for 200 users
Malicious users: 10
Normal users: 190

Feature columns: ['logon_count', 'failed_logon_count', 'after_hours_logon', 'usb_connect_count', 'usb_disconnect_count', 'email_sent_count', 'email_with_attachment', 'external_email_count', 'file_access_count', 'file_copy_count', 'http_request_count', 'job_search_visits', 'is_malicious']


## 3. Apply SMOTE for Class Balancing

In [19]:
# Prepare data
X = features_df.drop('is_malicious', axis=1)
y = features_df['is_malicious']

print("Original class distribution:")
print(f"  Normal: {(y == 0).sum()}")
print(f"  Malicious: {(y == 1).sum()}")

# Split data for final holdout evaluation
X_train_raw, X_test, y_train_raw, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("\nSplit sizes:")
print(f"  Train: {len(X_train_raw)}")
print(f"  Test: {len(X_test)}")
print(f"  Train normal: {(y_train_raw == 0).sum()}")
print(f"  Train malicious: {(y_train_raw == 1).sum()}")
print(f"  Test normal: {(y_test == 0).sum()}")
print(f"  Test malicious: {(y_test == 1).sum()}")

Original class distribution:
  Normal: 190
  Malicious: 10

Split sizes:
  Train: 160
  Test: 40
  Train normal: 152
  Train malicious: 8
  Test normal: 38
  Test malicious: 2


## 4. Train XGBoost Classifier

In [20]:
# Compare balancing strategies using repeated stratified CV
neg_count = (y_train_raw == 0).sum()
pos_count = (y_train_raw == 1).sum()
scale_pos_weight = neg_count / pos_count if pos_count > 0 else 1.0

base_params = dict(
    n_estimators=120,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_lambda=2.0,
    random_state=42,
    eval_metric='logloss'
)
conservative_params = dict(
    n_estimators=80,
    max_depth=3,
    learning_rate=0.04,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_lambda=3.0,
    random_state=42,
    eval_metric='logloss'
)

strategy_estimators = {
    'SMOTE + XGBoost': ImbPipeline(steps=[
        ('smote', SMOTE(random_state=42)),
        ('model', XGBClassifier(**base_params))
    ]),
    'Class-Weighted XGBoost': XGBClassifier(
        **base_params,
        scale_pos_weight=scale_pos_weight
    ),
    'SMOTE + Conservative XGBoost': ImbPipeline(steps=[
        ('smote', SMOTE(random_state=42)),
        ('model', XGBClassifier(**conservative_params))
    ])
}

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=42)
rows = []

for strategy_name, estimator in strategy_estimators.items():
    scores = cross_validate(
        estimator,
        X,
        y,
        cv=cv,
        scoring=['accuracy', 'f1', 'roc_auc', 'average_precision'],
        n_jobs=-1
    )
    rows.append({
        'strategy': strategy_name,
        'accuracy_mean': scores['test_accuracy'].mean(),
        'accuracy_std': scores['test_accuracy'].std(),
        'f1_mean': scores['test_f1'].mean(),
        'f1_std': scores['test_f1'].std(),
        'roc_auc_mean': scores['test_roc_auc'].mean(),
        'roc_auc_std': scores['test_roc_auc'].std(),
        'pr_auc_mean': scores['test_average_precision'].mean(),
        'pr_auc_std': scores['test_average_precision'].std()
    })

results_df = pd.DataFrame(rows).sort_values(
    by=['pr_auc_mean', 'f1_mean', 'accuracy_mean'], ascending=False
).reset_index(drop=True)

print("Model strategy comparison (Repeated 5x10 CV):")
print(results_df.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

# Prefer class-weighted model if top scores are tied at 4 decimal places
top = results_df.iloc[0]
tied = results_df[
    (results_df['pr_auc_mean'].round(4) == round(top['pr_auc_mean'], 4))
    & (results_df['f1_mean'].round(4) == round(top['f1_mean'], 4))
    & (results_df['accuracy_mean'].round(4) == round(top['accuracy_mean'], 4))
]
if 'Class-Weighted XGBoost' in tied['strategy'].values:
    best_strategy = 'Class-Weighted XGBoost'
else:
    best_strategy = top['strategy']

print(f"\nSelected strategy: {best_strategy}")

# Train selected strategy on training split and evaluate on untouched holdout set
if best_strategy == 'Class-Weighted XGBoost':
    xgb_model = XGBClassifier(**base_params, scale_pos_weight=scale_pos_weight)
    xgb_model.fit(X_train_raw, y_train_raw)
elif best_strategy == 'SMOTE + XGBoost':
    xgb_model = ImbPipeline(steps=[
        ('smote', SMOTE(random_state=42)),
        ('model', XGBClassifier(**base_params))
    ])
    xgb_model.fit(X_train_raw, y_train_raw)
else:
    xgb_model = ImbPipeline(steps=[
        ('smote', SMOTE(random_state=42)),
        ('model', XGBClassifier(**conservative_params))
    ])
    xgb_model.fit(X_train_raw, y_train_raw)

y_pred = xgb_model.predict(X_test)
y_prob = xgb_model.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)
pr_auc = average_precision_score(y_test, y_prob)

print("\nHoldout Evaluation (untouched test split):")
print(classification_report(y_test, y_pred, target_names=['Normal', 'Malicious']))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print(f"Accuracy: {acc:.4f}")
print(f"ROC-AUC: {auc:.4f}")
print(f"PR-AUC: {pr_auc:.4f}")

Model strategy comparison (Repeated 5x10 CV):
                    strategy  accuracy_mean  accuracy_std  f1_mean  f1_std  roc_auc_mean  roc_auc_std  pr_auc_mean  pr_auc_std
             SMOTE + XGBoost         0.9945        0.0115   0.9200  0.1833        1.0000       0.0000       1.0000      0.0000
      Class-Weighted XGBoost         0.9945        0.0115   0.9200  0.1833        1.0000       0.0000       1.0000      0.0000
SMOTE + Conservative XGBoost         0.9945        0.0115   0.9200  0.1833        0.9995       0.0037       0.9900      0.0700

Selected strategy: Class-Weighted XGBoost

Holdout Evaluation (untouched test split):
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00        38
   Malicious       1.00      1.00      1.00         2

    accuracy                           1.00        40
   macro avg       1.00      1.00      1.00        40
weighted avg       1.00      1.00      1.00        40

Confusion Matrix:
[[38  0]
 [ 0 

## 5. Train Isolation Forest for Anomaly Detection

In [21]:
# Train Isolation Forest on original (non-SMOTE) data
iso_forest = IsolationForest(
    n_estimators=100,
    contamination=0.1,  # Expect ~10% anomalies
    random_state=42
)

print("Training Isolation Forest...")
iso_forest.fit(X)

# Predict anomalies
anomaly_predictions = iso_forest.predict(X)
anomaly_scores = iso_forest.score_samples(X)

# Convert predictions: -1 (anomaly) to 1, 1 (normal) to 0
anomaly_labels = np.where(anomaly_predictions == -1, 1, 0)

print(f"\nAnomalies detected: {anomaly_labels.sum()}")
print(f"Normal samples: {(anomaly_labels == 0).sum()}")

Training Isolation Forest...

Anomalies detected: 20
Normal samples: 180


## 6. Save Models

In [22]:
# Create models directory
os.makedirs('../models', exist_ok=True)

# Save XGBoost model
with open('../models/xgboost_model.pkl', 'wb') as f:
    pickle.dump(xgb_model, f)
print("✓ XGBoost model saved to ../models/xgboost_model.pkl")

# Save Isolation Forest model
with open('../models/isolation_forest.pkl', 'wb') as f:
    pickle.dump(iso_forest, f)
print("✓ Isolation Forest model saved to ../models/isolation_forest.pkl")

# Save feature names for later use
feature_names = list(X.columns)
with open('../models/feature_names.pkl', 'wb') as f:
    pickle.dump(feature_names, f)
print("✓ Feature names saved to ../models/feature_names.pkl")

print("\n" + "="*60)
print("Model Training Complete!")
print("="*60)
print("\nNext steps:")
print("1. Start the backend: cd backend && uvicorn main:app --reload")
print("2. Start the frontend: cd frontend && npm run dev")
print("3. Navigate to http://localhost:5173")

✓ XGBoost model saved to ../models/xgboost_model.pkl
✓ Isolation Forest model saved to ../models/isolation_forest.pkl
✓ Feature names saved to ../models/feature_names.pkl

Model Training Complete!

Next steps:
1. Start the backend: cd backend && uvicorn main:app --reload
2. Start the frontend: cd frontend && npm run dev
3. Navigate to http://localhost:5173
